***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [7. 观测系统](7_0_introduction.ipynb)
    * 上一节： [7.7 传播效应](7_7_propagation_effects.ipynb)
    * 下一节： [7.9 系统温度、等效系统通量密度、观测日志与质量控制](7_9_system_temperature_sefd_observing_logs_qa.ipynb)

***


## 7.8 射频干扰（RFI）

RFI（Radio Frequency Interference）是射电干涉观测里最现实、也最难完全摆脱的问题之一。它不是来自宇宙，而是来自人类自己的发射系统：广播、通信、雷达、卫星、航空电子、开关电源，甚至观测站内部的电子设备都可能成为干扰源。与热噪声不同，RFI 往往不满足平稳高斯假设，因此它不仅会抬高噪声底，还会改变数据的统计结构、破坏相关器输出的分布假设，并让后续的成像和标定步骤面临新的偏差来源。

RFI 处理的关键并不只是“删掉坏点”。更准确地说，它是在污染抑制与灵敏度保留之间寻找平衡：标记太少，残余干扰会污染带通、相位和成像；标记太多，宝贵的积分时间和频率覆盖又会被无谓牺牲。成熟管线之所以会不断调节阈值、窗口与形态学操作，正是因为 RFI 的形态并不统一。


### 7.8.1 动态频谱中的 RFI 形态

在时频平面上，RFI 往往以几种典型形态出现：窄带连续线会长期占据少数频率通道，宽带突发会在很短的时间内抬高大段频带，扫频或啁啾结构会在频率轴上缓慢移动，而弱小的团簇事件则常常只有在统计上才会显出影响。真实观测里，这些形态经常叠加出现，因此单一阈值很难同时对所有场景都有效。

为了说明这一点，这里构造一个合成动态频谱，把几种最常见的 RFI 形态放进同一个数据块中，并保留一个真值掩膜作为对照。这样既可以观察 RFI 的时频纹理，也可以评估后续自动标记流程的召回率和误标率。


![混合形态的合成动态频谱](figures/rfi_dynamic_spectrum_and_mask.png)

**图 7.8.1** 左图是含有多种 RFI 形态的合成动态频谱，右图是真值占据掩膜。窄带连续线、宽带突发、扫频结构和局部团簇在同一数据块中并存，说明 RFI 识别必须同时考虑时间和频率两个方向。


### 7.8.2 一个基础但实用的自动标记流程

常见的自动标记方法很多，例如基于阈值传播的 SumThreshold、谱峰度（spectral kurtosis）、形态学检测以及更复杂的子空间方法。这里采用一个更基础、但仍然很有代表性的流程：先用时间中值估计每个频率通道的整体背景，再用频率滑动中值平滑带通，然后分别对通道级异常和时频像素级异常进行鲁棒阈值检测。对检测结果再做少量膨胀，可以把污染边缘一并纳入掩膜，因为真实 RFI 往往不是单个孤立像素，而是具有有限时间宽度和频率宽度的结构。

这里采用的鲁棒尺度估计是 MAD，即

$$
\sigma_{\rm MAD}=1.4826\,\mathrm{median}\,|x-\mathrm{median}(x)|.
$$

与普通标准差相比，MAD 对少量极端异常值更不敏感，因此很适合先验上已经知道“多数数据是正常的，少数点可能被污染”的场景。以此得到的鲁棒 z-score 可以写成

$$
z=\frac{D-B}{\sigma_{\rm MAD}},
$$

其中 $D$ 是观测数据，$B$ 是平滑背景。阈值并不是固定真理，而只是一个兼顾召回率与精确率的工程选择。


![鲁棒标记与带通恢复](figures/rfi_flagging_recovery.png)

**图 7.8.2** 左上为鲁棒 z-score，右上为自动标记掩膜，左下为标记前后的带通恢复，右下为真值占空比与实际标记占空比的对比。鲁棒统计的作用并不是消除所有异常，而是在保留绝大多数干净数据的同时，把主要污染源尽可能隔离出来。


这组结果清楚地表明，标记策略本身就是一个取舍问题。阈值较保守时，误标会减少，但残余污染可能增加；阈值较激进时，召回率会上升，但有效数据占比会下降。真正成熟的处理链一般不会只靠单一数值，而是会结合观测频段、天线环境、目标源强度和科学目标来调整参数。


### 7.8.3 干涉仪里的额外抑制：fringe stopping 与积分平均

干涉仪在相关时通常会对相位中心执行 fringe stopping，使目标天体源在相关后尽量保持慢变化或近似静止。若某个地面干扰源没有与这套几何补偿相匹配，它在相关后的残余振荡就会留下额外 fringe rate。对一个按

$$
V_{\rm RFI}(t)\propto e^{2\pi i f_{\rm fr} t}
$$

变化的干扰，在时间长度为 $T$ 的平均中，其响应幅度会按

$$
\left|\frac{1}{T}\int_0^T e^{2\pi i f_{\rm fr} t}\,dt\right|
=|\mathrm{sinc}(f_{\rm fr}T)|
$$

衰减。这里的 $\mathrm{sinc}$ 采用常用归一化定义。这个结果说明，fringe stopping 并不会让 RFI 自动消失，但它确实会对某些与天空条纹率不匹配的干扰施加额外的时间平均抑制。


![fringe stopping 对非匹配 fringe rate 的抑制](figures/rfi_fringe_stopping_attenuation.png)

**图 7.8.3** 不同 fringe rate 在随积分时间增长时的平均响应。相位中心天体源在 fringe stopping 后接近零 fringe rate，而地面 RFI 若残余 fringe rate 不为零，则会随积分时间被 sinc 包络部分压低。


这里需要特别强调两点。第一，fringe stopping 不是万能的 RFI 消除器，因为很多干扰仍可能与阵列几何保持部分相干，甚至在某些基线和时间段里显得非常稳定。第二，RFI 处理从来不是单层操作，而是从观测前的频谱规划、观测中的硬件抑制、相关前后的统计标记，到成像阶段对 flag 的正确传播，层层相扣。

因此，RFI 更适合被理解成一套跨越观测全流程的鲁棒统计问题，而不是一个可以在数据末端一次性修补掉的局部缺陷。它与前面几节讨论的 Jones 语言并不矛盾：只是这里的 Jones 链之外，又多了一层强非高斯污染和数据完整性管理的现实约束。
